# Toy Torch notebook

A tiny notebook proving that `training-manager` can supervise notebook-based
Torch training and collect structured telemetry.


In [ ]:
import os
import random
import time
from pathlib import Path

import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

try:
    from training_manager.sdk import TrainingReporter
    reporter = TrainingReporter(os.environ.get('TRAINING_MANAGER_DIR'))
except Exception as exc:
    reporter = None
    print(f'reporter unavailable: {exc}')

if reporter:
    reporter.stage('toy-bootstrap')

print('torch', torch.__version__)
print('device', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
raw = load_digits()
X = torch.tensor(raw.data, dtype=torch.float32) / 16.0
Y = torch.tensor(raw.target, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=SEED, stratify=Y
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

if reporter:
    reporter.stage('toy-data', train_rows=len(X_train), test_rows=len(X_test))

print('train rows', len(X_train), 'test rows', len(X_test))


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

if reporter:
    reporter.stage('toy-model')

print(model)


In [ ]:
EPOCHS = 5
ckpt_dir = Path('checkpoints')
ckpt_dir.mkdir(exist_ok=True)

if reporter:
    reporter.stage('toy-train', epochs=EPOCHS)

started = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total = 0
    correct = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total += xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()

    train_loss = total_loss / total
    train_acc = correct / total

    model.eval()
    test_total = 0
    test_correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            test_total += xb.size(0)
            test_correct += (logits.argmax(dim=1) == yb).sum().item()
    test_acc = test_correct / test_total

    elapsed = time.time() - started
    remaining_epochs = EPOCHS - epoch
    eta_seconds = (elapsed / epoch) * remaining_epochs if epoch else None

    print({
        'epoch': epoch,
        'train_loss': round(train_loss, 4),
        'train_acc': round(train_acc, 4),
        'test_acc': round(test_acc, 4),
    })

    if reporter:
        reporter.metric(step=epoch, train_loss=float(train_loss), train_acc=float(train_acc), test_acc=float(test_acc))
        reporter.progress(step=epoch, total_steps=EPOCHS, eta_seconds=None if eta_seconds is None else float(eta_seconds), stage='toy-train')

    ckpt_path = ckpt_dir / f'toy-digits-epoch-{epoch}.pt'
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
    }, ckpt_path)
    if reporter:
        reporter.checkpoint(str(ckpt_path), status='saved', epoch=epoch)


In [ ]:
if reporter:
    reporter.stage('toy-finished')

print('toy notebook complete')
